# Assistant biomédical à verdict calibré — Jour 1
### Cadrage, chargement des données, baseline zero-shot

Environnement cible : **Google Colab gratuit (GPU T4)**
Ce notebook couvre les tâches du Jour 1 du planning :
- Chargement et nettoyage de PubMedQA (`pqa_artificial` + `pqa_labeled`)
- Échantillonnage raisonnable pour Colab gratuit
- Baseline zero-shot (LLM prompté, sans fine-tuning)
- Sauvegarde des splits sur Google Drive pour les jours suivants


## 1. Vérification du GPU et installation des dépendances

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv


In [ ]:
!pip install -q datasets transformers accelerate sentence-transformers faiss-cpu evaluate scikit-learn peft bitsandbytes


## 2. Montage de Google Drive
On sauvegarde systématiquement les données préparées et les futurs checkpoints
dans Drive, car les sessions Colab gratuites peuvent se déconnecter à tout moment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/assistant_biomedical"
os.makedirs(PROJECT_DIR, exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/data", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{PROJECT_DIR}/results", exist_ok=True)
print("Répertoires prêts dans :", PROJECT_DIR)


## 3. Reproductibilité
On fixe toutes les seeds pour garantir des résultats reproductibles d'une session à l'autre.

In [ ]:
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device utilisé :", device)


## 4. Chargement de PubMedQA
- `pqa_labeled` : 1 000 exemples annotés par des experts → notre jeu de **test propre**
- `pqa_artificial` : ~211 000 exemples générés automatiquement → source d'entraînement,
  dont on échantillonne ~20 000 exemples comme prévu au cahier des charges.

In [ ]:
from datasets import load_dataset

labeled = load_dataset("pubmed_qa", "pqa_labeled")
artificial = load_dataset("pubmed_qa", "pqa_artificial")

print(labeled)
print(artificial)


In [ ]:
# Aperçu d'un exemple pour vérifier la structure
example = labeled["train"][0]
for k, v in example.items():
    print(f"--- {k} ---")
    print(v)
    print()


## 5. Nettoyage et mise en forme

On construit un format unique et propre :
- `question`
- `context` (concaténation des sections du `CONTEXTS` de l'abstract)
- `label` (yes / no / maybe)

On retire les exemples avec un contexte vide ou un label manquant.

In [ ]:
def flatten_example(ex):
    contexts = ex.get("context", {})
    # Le champ context est un dict avec une clé 'contexts' listant les sections
    if isinstance(contexts, dict) and "contexts" in contexts:
        context_text = " ".join(contexts["contexts"])
    else:
        context_text = str(contexts)

    return {
        "question": ex["question"],
        "context": context_text.strip(),
        "label": ex["final_decision"].strip().lower(),
    }

def clean_dataset(ds_split):
    flat = [flatten_example(ex) for ex in ds_split]
    flat = [f for f in flat if f["context"] and f["label"] in ("yes", "no", "maybe")]
    return flat

labeled_clean = clean_dataset(labeled["train"])
artificial_clean = clean_dataset(artificial["train"])

print("Exemples labeled (experts) après nettoyage :", len(labeled_clean))
print("Exemples artificial après nettoyage :", len(artificial_clean))


In [ ]:
from collections import Counter

print("Distribution des labels — labeled (test):", Counter([x["label"] for x in labeled_clean]))
print("Distribution des labels — artificial (train, avant échantillonnage):",
      Counter([x["label"] for x in artificial_clean]))


## 6. Échantillonnage pour Colab gratuit

Conformément au cahier des charges : on prend ~20 000 exemples du sous-ensemble
`artificial` pour l'entraînement, en conservant si possible la proportion des classes
d'origine (échantillonnage stratifié simple).

`pqa_labeled` est conservé intégralement comme test final (il ne sert jamais à
l'entraînement).

In [ ]:
import random

def stratified_sample(data, n_total, seed=SEED):
    random.seed(seed)
    by_label = {"yes": [], "no": [], "maybe": []}
    for ex in data:
        by_label[ex["label"]].append(ex)

    total = len(data)
    sampled = []
    for label, items in by_label.items():
        random.shuffle(items)
        n_label = round(n_total * len(items) / total)
        sampled.extend(items[:n_label])

    random.shuffle(sampled)
    return sampled

TRAIN_SAMPLE_SIZE = 20000
artificial_sampled = stratified_sample(artificial_clean, TRAIN_SAMPLE_SIZE)

print("Taille de l'échantillon d'entraînement :", len(artificial_sampled))
print("Distribution après échantillonnage :", Counter([x["label"] for x in artificial_sampled]))


## 7. Split entraînement / validation

On découpe l'échantillon artificiel en train/validation (90/10).
Le test final reste exclusivement `pqa_labeled`.

In [ ]:
split_idx = int(0.9 * len(artificial_sampled))
train_data = artificial_sampled[:split_idx]
val_data = artificial_sampled[split_idx:]
test_data = labeled_clean  # jamais utilisé avant l'évaluation finale

print(f"Train : {len(train_data)} | Validation : {len(val_data)} | Test (expert) : {len(test_data)}")


## 8. Sauvegarde des splits sur Drive
Pour ne pas refaire ce travail à chaque session, on sauvegarde les splits en JSON.

In [ ]:
import json

def save_json(data, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

save_json(train_data, f"{PROJECT_DIR}/data/train.json")
save_json(val_data, f"{PROJECT_DIR}/data/val.json")
save_json(test_data, f"{PROJECT_DIR}/data/test_expert.json")

print("Splits sauvegardés dans", f"{PROJECT_DIR}/data/")


## 9. Baseline zero-shot (obligatoire avant tout fine-tuning)

On prompte un petit LLM instruct (`Qwen2.5-1.5B-Instruct`) **sans aucun fine-tuning**
pour répondre yes/no/maybe à partir de la question + contexte. Cette baseline sert de
point de comparaison pour mesurer la valeur ajoutée du fine-tuning LoRA (Jour 2).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

BASELINE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer_zs = AutoTokenizer.from_pretrained(BASELINE_MODEL)
model_zs = AutoModelForCausalLM.from_pretrained(
    BASELINE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto",
)


In [ ]:
def build_prompt(question, context):
    return (
        "Tu es un assistant biomédical. Réponds uniquement par un seul mot : "
        "yes, no ou maybe, selon que le contexte confirme, infirme, ou ne permet "
        "pas de conclure clairement sur la question.\n\n"
        f"Contexte : {context}\n\n"
        f"Question : {question}\n\n"
        "Réponse (un seul mot parmi yes / no / maybe) :"
    )

def zero_shot_predict(question, context, max_new_tokens=5):
    prompt = build_prompt(question, context)
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer_zs.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model_zs.device)

    with torch.no_grad():
        output = model_zs.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer_zs.eos_token_id,
        )

    generated = tokenizer_zs.decode(
        output[0][inputs.shape[-1]:], skip_special_tokens=True
    ).strip().lower()

    for label in ("yes", "no", "maybe"):
        if label in generated:
            return label
    return "maybe"  # repli prudent si la génération est ambiguë


In [ ]:
# Test rapide sur 5 exemples avant de lancer l'évaluation complète
for ex in test_data[:5]:
    pred = zero_shot_predict(ex["question"], ex["context"])
    print(f"Vrai label : {ex['label']:6s} | Prédit : {pred:6s} | Question : {ex['question'][:70]}...")


## 10. Évaluation complète de la baseline sur le test expert

⚠️ Sur T4 gratuit, 1 000 exemples en génération peuvent prendre plusieurs minutes.
On affiche une barre de progression et on sauvegarde les prédictions au fur et à mesure.

In [ ]:
from tqdm import tqdm

baseline_predictions = []
for ex in tqdm(test_data, desc="Baseline zero-shot"):
    pred = zero_shot_predict(ex["question"], ex["context"])
    baseline_predictions.append({
        "question": ex["question"],
        "true_label": ex["label"],
        "predicted_label": pred,
    })

save_json(baseline_predictions, f"{PROJECT_DIR}/results/baseline_zero_shot_predictions.json")
print("Prédictions baseline sauvegardées.")


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report

y_true = [p["true_label"] for p in baseline_predictions]
y_pred = [p["predicted_label"] for p in baseline_predictions]

acc = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print(f"Baseline zero-shot — Accuracy : {acc:.4f} | Macro-F1 : {macro_f1:.4f}\n")
print(classification_report(y_true, y_pred, digits=3))


## 11. Bilan du Jour 1

- ✅ Données PubMedQA chargées, nettoyées et échantillonnées (20 000 train / val / 1 000 test expert)
- ✅ Splits sauvegardés sur Drive (`data/train.json`, `data/val.json`, `data/test_expert.json`)
- ✅ Baseline zero-shot établie et chiffrée (accuracy, macro-F1)

**Prochaine étape (Jour 2)** : fine-tuning LoRA de `PubMedBERT-base-uncased-abstract`
sur `train.json`, avec `val.json` pour le suivi, puis comparaison directe avec cette
baseline zero-shot.